# Couche -> Bronze

Exécuter les cellules dans l'ordre après la collecte des données et le démarrage de RustFS. Une nouvelle exécution actualise les objets portant la même clé.

### 1 - Préparer l'environnement

Installer les bibliothèques utilisées par le notebook dans le noyau Jupyter actif.

In [1]:
%pip install -q boto3 pandas pyarrow python-dotenv

Note: you may need to restart the kernel to use updated packages.


### 2 - Accéder au bucket Bronze

In [2]:
import os
from pathlib import Path
from tempfile import TemporaryFile

import boto3
import pandas as pd
import pyarrow.parquet as pq
from botocore.config import Config
from dotenv import load_dotenv

current_dir = Path.cwd()
project_dir = next(
    (directory for directory in (current_dir, *current_dir.parents)
     if (directory / "compose.yaml").is_file()),
    None,
)
if project_dir is None:
    raise FileNotFoundError("Ouvrir le notebook depuis le dépôt du projet.")

load_dotenv(project_dir / ".env")
required_variables = ("S3_ENDPOINT", "S3_ACCESS_KEY", "S3_SECRET_KEY")
missing_variables = [name for name in required_variables if not os.getenv(name)]
if missing_variables:
    raise RuntimeError("Variables S3 manquantes : " + ", ".join(missing_variables))

BUCKET_NAME = "bronze"
DATA_DIR = project_dir / "data"

s3 = boto3.client(
    "s3",
    endpoint_url=os.environ["S3_ENDPOINT"],
    aws_access_key_id=os.environ["S3_ACCESS_KEY"],
    aws_secret_access_key=os.environ["S3_SECRET_KEY"],
    region_name="us-east-1",
    config=Config(signature_version="s3v4", s3={"addressing_style": "path"}),
)

buckets = {bucket["Name"] for bucket in s3.list_buckets()["Buckets"]}
if BUCKET_NAME not in buckets:
    s3.create_bucket(Bucket=BUCKET_NAME)

print(f"Bucket « {BUCKET_NAME} » disponible.")

Bucket « bronze » disponible.


### 3 - Classer et envoyer les fichiers

In [3]:
SOURCES = {
    "instacart": ("instacart",),
    "openfoodfacts": ("openfoodfacts", "open_food_facts"),
    "openprices": ("openprices", "open_prices"),
    "rappelconso": ("rappelconso", "rappel_conso"),
}

def detect_source(path):
    parts = [part.lower() for part in path.parts]
    for source, keywords in SOURCES.items():
        if any(keyword in part for keyword in keywords for part in parts):
            return source
    return None

parquet_files = sorted(DATA_DIR.rglob("*.parquet"))
if not parquet_files:
    print("Aucun fichier Parquet dans data/. Lancer la collecte pour alimenter le bucket.")

destinations = {}
report = []
for file in parquet_files:
    relative_path = file.relative_to(DATA_DIR)
    source = detect_source(relative_path)
    if source is None:
        report.append((relative_path.as_posix(), "—", "Source non identifiée"))
        continue

    key = f"{source}/raw/{file.name}"
    if key in destinations:
        previous_file = destinations[key].relative_to(DATA_DIR)
        raise ValueError(f"Clé Bronze en double : {key} ({previous_file} et {relative_path})")

    destinations[key] = file
    report.append((relative_path.as_posix(), key, "Importé"))

for key, file in destinations.items():
    s3.upload_file(str(file), BUCKET_NAME, key)

if report:
    print(f"{len(destinations)} fichier(s) importé(s) sur {len(parquet_files)} détecté(s).")
    display(pd.DataFrame(report, columns=["Fichier local", "Clé Bronze", "Statut"]))

9 fichier(s) importé(s) sur 9 détecté(s).


,Fichier local,Clé Bronze,Statut
0,instacart/aisles.parquet,instacart/raw/aisles.parquet,Importé
1,instacart/departments.parquet,instacart/raw/departments.parquet,Importé
2,instacart/order_products__prior.parquet,instacart/raw/order_products__prior.parquet,Importé
3,instacart/order_products__train.parquet,instacart/raw/order_products__train.parquet,Importé
4,instacart/orders.parquet,instacart/raw/orders.parquet,Importé
5,instacart/products.parquet,instacart/raw/products.parquet,Importé
6,openfoodfacts/produits.parquet,openfoodfacts/raw/produits.parquet,Importé
7,openprices/page.parquet,openprices/raw/page.parquet,Importé
8,rappelconso/rappels.parquet,rappelconso/raw/rappels.parquet,Importé


### 4 - Vérifier le contenu du bucket

L'inventaire inclut tous les fichiers Parquet présents dans Bronze, y compris ceux issus des imports précédents. Leur taille vient du stockage et le nombre de lignes et de colonnes est lu dans les métadonnées Parquet.

In [ ]:
inventory = []
paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket=BUCKET_NAME):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if not key.endswith(".parquet"):
            continue

        with TemporaryFile() as parquet_data:
            s3.download_fileobj(BUCKET_NAME, key, parquet_data)
            parquet_data.seek(0)
            parquet_file = pq.ParquetFile(parquet_data)
            inventory.append((
                key,
                round(obj["Size"] / 1024**2, 2),
                parquet_file.metadata.num_rows,
                len(parquet_file.schema_arrow),
            ))

if inventory:
    display(pd.DataFrame(sorted(inventory), columns=["Clé Bronze", "Taille (Mio)", "Lignes", "Colonnes"]))
else:
    print("Le bucket Bronze ne contient aucun fichier Parquet.")

,Clé Bronze,Taille (Mio),Lignes,Colonnes
0,instacart/raw/aisles.parquet,0.00,134,2
1,instacart/raw/departments.parquet,0.00,21,2
2,instacart/raw/order_products__prior.parquet,133.87,32434489,4
3,instacart/raw/order_products__train.parquet,5.73,1384617,4
4,instacart/raw/orders.parquet,36.66,3421083,7
5,instacart/raw/products.parquet,1.47,49688,4
6,openfoodfacts/raw/produits.parquet,0.10,100,15
7,openprices/raw/page.parquet,0.02,1,2
8,rappelconso/raw/rappels.parquet,6.23,19260,34
